In [1]:
import numpy as np
from brian2 import *
from sklearn.datasets import fetch_openml

In [17]:
n_input = 784
n_e = 400
n_i = n_e 
time_per_img = 350 * ms  # Garg et al. presented images for 350ms [cite: 332]
target_weight = 78.0    
max_delay = 10 * ms

# --- VDSP Input Layer Parameters (Garg et al. Table 1) ---
lr_vdsp = 0.05              # Learning rate [cite: 412]
tau_in = 30 * ms            # Leak time constant [cite: 411]
v_rest_in = 0 * volt        # Resting voltage [cite: 411]
v_reset_in = -1.0 * volt    # Reset voltage [cite: 411]
v_thresh_in = 1.0 * volt    # Threshold voltage [cite: 411]
refrac_in = 5 * ms          # Refractory period [cite: 411]
input_bias = 0.5 * volt     # Bias to penalize background pixels [cite: 411]

# --- Diehl & Cook Excitatory/Inhibitory Parameters ---
E_rest = -65. * mV 
I_rest = -60. * mV 
E_reset = -65. * mV
I_reset = -45. * mV
E_thresh = -52. * mV
I_thresh = -40. * mV
E_refrac = 5. * ms
I_refrac = 2. * ms
E_exc = 0.0 * mV
E_inh = -100.0 * mV

init_theta = 0.0 * mV
d_theta = 0.05 * mV
theta_offset = 20.0 * mV 

w_ei = 10.4
w_ie = 17.0
wmax = 1.0

tau_e = 100 * ms       
tau_i = 10 * ms        
tau_ge = 1 * ms        
tau_gi = 2 * ms        
tau_theta = 1e7 * ms

In [18]:
# Input Layer (LIF neurons driven by pixel values)
eqs_in = '''
dv/dt = ((v_rest_in - v) + I_stim + input_bias) / tau_in : volt (unless refractory)
I_stim : volt  
'''

# Excitatory Neurons
eqs_e = '''
dv/dt = ((E_rest - v) + ge*(E_exc - v) + gi*(E_inh - v)) / tau_e : volt (unless refractory)
dge/dt = -ge / tau_ge : 1
dgi/dt = -gi / tau_gi : 1
dtheta/dt = -theta / tau_theta : volt
v_thresh = E_thresh + theta - theta_offset: volt
'''

# Inhibitory Neurons
eqs_i = '''
dv/dt = ((I_rest - v) + ge*(E_exc - v) + gi*(E_inh - v)) / tau_i : volt (unless refractory)
dge/dt = -ge / tau_ge : 1
dgi/dt = -gi / tau_gi : 1
'''

# VDSP Synapse Equations
eqs_stdp_vdsp = '''
w : 1
'''

eqs_stdp_pre_vdsp = '''
ge_post += w
'''
eqs_stdp_post_vdsp = '''
v_norm = v_pre / volt
dw_pot = (wmax - w) * (exp(-v_norm) - 1.0) * lr_vdsp * int(v_norm < 0)
dw_dep = -w * (exp(v_norm) - 1.0) * lr_vdsp * int(v_norm >= 0)
w = clip(w + dw_pot + dw_dep, 0.0, wmax)
'''

In [19]:
def build_network():
    # 1. Input Layer
    inp_group = NeuronGroup(n_input, eqs_in, threshold='v > v_thresh_in', 
                            reset='v = v_reset_in', refractory=refrac_in, 
                            method='euler', name='inp')
    inp_group.v = v_rest_in
    inp_group.I_stim = 0 * volt
    
    # 2. Excitatory Layer
    exc_group = NeuronGroup(n_e, eqs_e, threshold='v > v_thresh', 
                            reset='v = E_reset; theta += d_theta', 
                            refractory=E_refrac, method='euler', name='exc')
    exc_group.v = E_rest
    exc_group.theta = init_theta
    
    # 3. Inhibitory Layer
    inh_group = NeuronGroup(n_i, eqs_i, threshold='v > I_thresh', 
                            reset='v = I_reset', refractory=I_refrac, 
                            method='euler', name='inh')
    inh_group.v = I_rest
    
    # 4. Input -> Excitatory (VDSP Plasticity)
    S_inp_exc = Synapses(inp_group, exc_group, model=eqs_stdp_vdsp,
                         on_pre=eqs_stdp_pre_vdsp, on_post=eqs_stdp_post_vdsp, 
                         name='s_inp_exc')
    S_inp_exc.connect(p=1.0)
    S_inp_exc.w = 'rand() * wmax'
    S_inp_exc.delay = 'rand() * max_delay'

    # 5. Excitatory -> Inhibitory (1-to-1)
    S_exc_inh = Synapses(exc_group, inh_group, on_pre='ge_post += w_ei', name='s_exc_inh')
    S_exc_inh.connect(j='i')

    # 6. Inhibitory -> Excitatory (Winner-Take-All lateral inhibition)
    S_inh_exc = Synapses(inh_group, exc_group, on_pre='gi_post += w_ie', name='s_inh_exc')
    S_inh_exc.connect(condition='i != j')

    # Monitors
    spike_monitor = SpikeMonitor(exc_group, name='sp_exc')
        
    # Weight Normalization
    @network_operation(dt=time_per_img)
    def normalize_weights():
        w_array = S_inp_exc.w[:]
        col_sums = np.bincount(S_inp_exc.j, weights=w_array, minlength=n_e)
        col_sums[col_sums == 0] = 1.0 
        scale_factors = target_weight / col_sums
        S_inp_exc.w = w_array * scale_factors[S_inp_exc.j]

    net = Network(inp_group, exc_group, inh_group, S_inp_exc, S_exc_inh, S_inh_exc, 
                  spike_monitor, normalize_weights)
    
    return net, inp_group, spike_monitor

In [14]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='liac-arff')
X, y = mnist.data, mnist.target
num_training_images = 5

WARNING    'i' is an internal variable of group 's_exc_inh', but also exists in the run namespace with the value 0. The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]
WARNING    'i' is an internal variable of group 's_inh_exc', but also exists in the run namespace with the value 0. The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]


In [20]:
net, inp_group, spike_monitor = build_network()

WARNING    'i' is an internal variable of group 's_exc_inh', but also exists in the run namespace with the value 4. The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]
WARNING    'i' is an internal variable of group 's_inh_exc', but also exists in the run namespace with the value 4. The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]


In [21]:
for i in range(num_training_images):
    # 1. Fetch the image and normalize it
    # MNIST pixels from fetch_openml are 0-255. We scale them to 0.0 - 1.0.
    image = X[i]
    pixel_values = np.clip(image / 255.0, 0.0, 1.0)
    
    # 2. Feed the pixels directly into the input neurons as voltage/current!
    # Bright pixels (~1.0V) cause rapid firing, driving potentiation.
    # Dark pixels (~0.0V) rely on input_bias, causing depression.
    inp_group.I_stim = pixel_values * volt
    
    # 3. Run the simulation for the duration of this image
    label = y[i]
    print(f"  Training on image {i+1}/{num_training_images} (Label: {label})...")
    net.run(time_per_img)
    
    # Optional: Add a brief "resting" phase with 0 stimulus if you want 
    # to let the membrane potentials settle between images to prevent bleed-over.
    # inp_group.I_stim = 0 * volt
    # net.run(50 * ms)

print("Training complete. Total spikes in excitatory layer:", spike_monitor.num_spikes)

  Training on image 1/5 (Label: 5)...
  Training on image 2/5 (Label: 0)...
  Training on image 3/5 (Label: 4)...
  Training on image 4/5 (Label: 1)...
  Training on image 5/5 (Label: 9)...
Training complete. Total spikes in excitatory layer: 1227
